<a href="https://colab.research.google.com/github/JJcoders00/slm/blob/main/JJ_Coders_AI_From_Scratch_1st-notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JJ Coders - Proprietary AI Model From Scratch
### End-to-End Pipeline: Dataset -> Custom BPE Tokenizer -> Model Architecture -> Pre-Training -> Interactive Generation

This notebook builds a custom Small Language Model (SLM) from the ground up on Google Colab (Free T4 GPU) with full persistence to Google Drive.

## 1. System Setup & Google Drive Mounting
Mount Google Drive to ensure checkpoints and model weights persist across disconnects.

In [1]:
import os
import torch

# Check GPU availability
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"Allocated VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("Warning: Running on CPU. Switch runtime to T4 GPU under Runtime > Change runtime type.")

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create checkpoint directory in Google Drive
SAVE_DIR = "/content/drive/MyDrive/JJ_AI_Project"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Model and checkpoint storage ready at: {SAVE_DIR}")

PyTorch Version: 2.11.0+cu128
Active GPU: Tesla T4
Allocated VRAM: 15.64 GB
Mounted at /content/drive
Model and checkpoint storage ready at: /content/drive/MyDrive/JJ_AI_Project


## 2. Install Lightweight Dependencies & Download Foundation Dataset
We use `tokenizers` and `datasets` to fetch a clean, high-signal training corpus (TinyStories).

In [2]:
!pip install -q tokenizers datasets

from datasets import load_dataset

DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)
raw_text_path = os.path.join(DATA_DIR, "corpus.txt")

print("Downloading high-density foundational dataset (TinyStories subset)...")
dataset = load_dataset("roneneldan/TinyStories", split="train[:50000]")

# Write raw text to corpus.txt
with open(raw_text_path, "w", encoding="utf-8") as f:
    for item in dataset:
        text = item["text"].strip()
        if text:
            f.write(text + "\n<|endoftext|>\n")

print(f"Corpus created successfully: {os.path.getsize(raw_text_path) / 1e6:.2f} MB")

README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…): reconstructing file:   0%|          |  0.00B /  249MB            

data/train-00000-of-00004-2d5a1467fff108(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-5852b56a2bd28f(…): reconstructing file:   0%|          |  0.00B /  248MB            

data/train-00001-of-00004-5852b56a2bd28f(…): downloading bytes:           |  0.00B            

...

data/validation-00000-of-00001-869c898b5(…): reconstructing file:   0%|          |  0.00B / 9.99MB            

data/validation-00000-of-00001-869c898b5(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Corpus created successfully: 45.37 MB


## 3. Train Custom Byte-Pair Encoding (BPE) Tokenizer
Trains a dedicated tokenizer with a compact vocabulary of 4,096 tokens.

In [3]:
from tokenizers import ByteLevelBPETokenizer

TOKENIZER_DIR = os.path.join(SAVE_DIR, "jj_tokenizer")
os.makedirs(TOKENIZER_DIR, exist_ok=True)

tokenizer = ByteLevelBPETokenizer()
tokenizer.train(
    files=[raw_text_path],
    vocab_size=4096,
    min_frequency=2,
    special_tokens=["<pad>", "<s>", "</s>", "<unk>", "<|endoftext|>"]
)

tokenizer.save_model(TOKENIZER_DIR)
print(f"Custom JJ Tokenizer trained and saved to: {TOKENIZER_DIR}")

Custom JJ Tokenizer trained and saved to: /content/drive/MyDrive/JJ_AI_Project/jj_tokenizer


## 4. Binary Tokenization (`train.bin`)
Converts all text into a flat `uint16` binary array to allow zero-RAM memory-mapped streaming.

In [4]:
import numpy as np

print("Encoding corpus into binary tokens...")
with open(raw_text_path, "r", encoding="utf-8") as f:
    text = f.read()

encoded_tokens = tokenizer.encode(text).ids
token_arr = np.array(encoded_tokens, dtype=np.uint16)

bin_path = os.path.join(DATA_DIR, "train.bin")
token_arr.tofile(bin_path)
print(f"Total training tokens: {len(token_arr):,}")
print(f"Binary file saved at: {bin_path} ({os.path.getsize(bin_path) / 1e6:.2f} MB)")

Encoding corpus into binary tokens...
Total training tokens: 11,458,199
Binary file saved at: /content/data/train.bin (22.92 MB)


## 5. Define JJ Coders Model Architecture (PyTorch)
Modern, efficient Transformer with RMSNorm, RoPE, SwiGLU, and Weight-Tied Recurrent Depth.

In [5]:
import math
import torch.nn as nn
import torch.nn.functional as F

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight

def precompute_rope_freqs(dim: int, max_seq_len: int, theta: float = 10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    return torch.polar(torch.ones_like(freqs), freqs)

def apply_rotary_emb(xq, xk, freqs_cis):
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
    freqs_cis = freqs_cis[:xq.shape[1], :].to(xq.device).view(1, xq.shape[1], 1, -1)
    xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(3)
    xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(3)
    return xq_out.type_as(xq), xk_out.type_as(xk)

class SwiGLUMLP(nn.Module):
    def __init__(self, dim: int, hidden_dim: int):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(dim, hidden_dim, bias=False)
        self.w3 = nn.Linear(hidden_dim, dim, bias=False)

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))

class TransformerBlock(nn.Module):
    def __init__(self, dim: int, n_heads: int):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)
        self.norm1 = RMSNorm(dim)
        self.norm2 = RMSNorm(dim)
        self.mlp = SwiGLUMLP(dim, int(dim * 2.67))

    def forward(self, x, freqs_cis):
        B, S, D = x.shape
        norm_x = self.norm1(x)
        q = self.q_proj(norm_x).view(B, S, self.n_heads, self.head_dim)
        k = self.k_proj(norm_x).view(B, S, self.n_heads, self.head_dim)
        v = self.v_proj(norm_x).view(B, S, self.n_heads, self.head_dim)
        q, k = apply_rotary_emb(q, k, freqs_cis)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
        attn_out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, S, D)
        h = x + self.out_proj(attn_out)
        return h + self.mlp(self.norm2(h))

class JJCodersModel(nn.Module):
    def __init__(self, vocab_size=4096, dim=384, n_heads=6, n_layers=4, recurrent_steps=2, max_seq_len=512):
        super().__init__()
        self.recurrent_steps = recurrent_steps
        self.embed = nn.Embedding(vocab_size, dim)
        self.blocks = nn.ModuleList([TransformerBlock(dim, n_heads) for _ in range(n_layers)])
        self.final_norm = RMSNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)
        self.embed.weight = self.lm_head.weight
        self.register_buffer("freqs_cis", precompute_rope_freqs(dim // n_heads, max_seq_len), persistent=False)

    def forward(self, input_ids, targets=None):
        x = self.embed(input_ids)
        for _ in range(self.recurrent_steps):
            for block in self.blocks:
                x = block(x, self.freqs_cis)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=100, temperature=0.8, top_k=40):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = input_ids if input_ids.size(1) <= 512 else input_ids[:, -512:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat((input_ids, idx_next), dim=1)
        return input_ids

print("JJ Coders Model Architecture defined successfully.")

JJ Coders Model Architecture defined successfully.


## 6. Training Engine with Persistent Auto-Checkpoints
Trains on the T4 GPU with mixed precision and saves progress directly to Google Drive.

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
CHECKPOINT_PATH = os.path.join(SAVE_DIR, "jj_model_checkpoint.pt")

# Model parameters (~18M physical params with 36M effective depth)
model = JJCodersModel(
    vocab_size=4096,
    dim=384,
    n_heads=6,
    n_layers=4,
    recurrent_steps=2,
    max_seq_len=512
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Physical Parameters: {total_params / 1e6:.2f}M | Effective Depth: 8 layers")

# Data loader function (Memory Mapped)
def get_batch(bin_file, batch_size=32, seq_len=256):
    data = np.memmap(bin_file, dtype=np.uint16, mode='r')
    ix = torch.randint(len(data) - seq_len - 1, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+seq_len]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+seq_len]).astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

# Optimizer & Scaler
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler()

# Auto-resume from Google Drive if checkpoint exists
start_step = 0
if os.path.exists(CHECKPOINT_PATH):
    print(f"Resuming training from checkpoint: {CHECKPOINT_PATH}")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scaler.load_state_dict(ckpt['scaler_state_dict'])
    start_step = ckpt['step'] + 1
    print(f"Resumed from step {start_step} (Saved Loss: {ckpt['loss']:.4f})")
else:
    print("Starting fresh training run.")

# Training Loop
max_steps = 2000
eval_interval = 200
save_interval = 400

model.train()
print(f"Beginning training for {max_steps} steps...")

for step in range(start_step, max_steps):
    xb, yb = get_batch(bin_path, batch_size=32, seq_len=256)
    optimizer.zero_grad(set_to_none=True)

    with torch.cuda.amp.autocast(dtype=torch.float16):
        logits, loss = model(xb, targets=yb)

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

    if (step + 1) % eval_interval == 0 or step == max_steps - 1:
        print(f"Step [{step+1}/{max_steps}] | Loss: {loss.item():.4f}")

    if (step + 1) % save_interval == 0 or step == max_steps - 1:
        torch.save({
            'step': step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'loss': loss.item()
        }, CHECKPOINT_PATH)
        print(f"--> Checkpoint saved to Google Drive at step {step+1}")

print("Training run finished successfully.")

Physical Parameters: 8.66M | Effective Depth: 8 layers


/tmp/ipykernel_1320/2654959808.py:27: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_1320/2654959808.py:54: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):


Starting fresh training run.
Beginning training for 2000 steps...
Step [200/2000] | Loss: 3.5223
Step [400/2000] | Loss: 3.0236
--> Checkpoint saved to Google Drive at step 400
Step [600/2000] | Loss: 2.8118
Step [800/2000] | Loss: 2.5388
--> Checkpoint saved to Google Drive at step 800
Step [1000/2000] | Loss: 2.5003
Step [1200/2000] | Loss: 2.3971
--> Checkpoint saved to Google Drive at step 1200
Step [1400/2000] | Loss: 2.2529
Step [1600/2000] | Loss: 2.1537
--> Checkpoint saved to Google Drive at step 1600
Step [1800/2000] | Loss: 2.0693
Step [2000/2000] | Loss: 2.0777
--> Checkpoint saved to Google Drive at step 2000
Training run finished successfully.


## 7. Interactive Test & Text Generation
Test your trained model with custom prompts.

In [7]:
prompt_text = "Once upon a time in a small village, there was a"
input_ids = torch.tensor([tokenizer.encode(prompt_text).ids], device=device)

generated_ids = model.generate(input_ids, max_new_tokens=80, temperature=0.7, top_k=30)
output_text = tokenizer.decode(generated_ids[0].tolist())

print("--- PROMPT ---")
print(prompt_text)
print("\n--- MODEL OUTPUT ---")
print(output_text)

--- PROMPT ---
Once upon a time in a small village, there was a

--- MODEL OUTPUT ---
Once upon a time in a small village, there was a little girl named Lily. She loved to play outside in the sunshine with her toys. One day, she found a big jar of milk in the ground. She wanted to make it look pretty, so she asked her mom if she could help her clean the jar.

Lily's mom said, "Lily, you need to clean your hands." Lily gave her the jar of milk and went inside
